In [4]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-e6b94c71-1db7-5ad2-a95a-9de8bd50bb1a"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import time
from tqdm.auto import tqdm 


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.5.1
CUDA available: False
Using device: cpu


In [5]:

LEARNING_RATE = 0.0005  
BATCH_SIZE = 256        
NUM_EPOCHS = 30         
NUM_CLASSES = 12


TRAIN_DATA_FILE = 'train.h5'
TEST_DATA_FILE = 'test.h5'
TEST_LABELS_FILE = 'test_labels.csv'

In [6]:
def load_and_preprocess_data():
    """Loads, concatenates, transposes, and scales all data."""
    
    print("Loading labels...")
    
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        
        y_train = np.array(hdf.get('Class/1'))


    y_test = pd.read_csv(TEST_LABELS_FILE, header=None).values.squeeze()

    print("Loading and concatenating training spectra (this may take 1-2 minutes)...")
    
    all_train_spectra = []
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        train_keys = [f"{i:03d}" for i in range(1, 101)] # Keys '001' to '100'
        for key in tqdm(train_keys, desc="Training data"):
            data_block = np.array(hdf[f'Spectra/{key}'])
            #  Transpose from (40002, 500) to (500, 40002)
            all_train_spectra.append(data_block.T)
    
    X_train_raw = np.concatenate(all_train_spectra, axis=0)

    print("Loading and concatenating test spectra...")
    # Load test spectra (X_test)
    all_test_spectra = []
    with h5py.File(TEST_DATA_FILE, 'r') as hdf:
        test_keys = ['1', '2'] # Keys '1' and '2'
        for key in tqdm(test_keys, desc="Test data    "):
            data_block = np.array(hdf[f'UNKNOWN/{key}'])
            # Transpose from (40002, 10000) to (10000, 40002)
            all_test_spectra.append(data_block.T)
            
    X_test_raw = np.concatenate(all_test_spectra, axis=0)

    print(f"Raw X_train shape: {X_train_raw.shape}, Raw y_train shape: {y_train.shape}")
    print(f"Raw X_test shape: {X_test_raw.shape}, Raw y_test shape: {y_test.shape}")

    print("Scaling data (this may take a moment)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    print("Data loading and preprocessing complete.")
    return X_train_scaled, y_train, X_test_scaled, y_test

In [7]:
class LibsDataset(Dataset):
    """Custom PyTorch Dataset for LIBS spectra."""
    def __init__(self, data, labels):
    
        # (N, 40002) -> (N, 1, 40002)
        self.X = torch.tensor(data, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(labels, dtype=torch.long)
        
        
        # 0-indexed (0 to 11)
    
        self.y = self.y - 1 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# LOAD THE DATA
X_train, y_train, X_test, y_test = load_and_preprocess_data()

print("\nCreating PyTorch Datasets...")
train_dataset = LibsDataset(X_train, y_train)
test_dataset = LibsDataset(X_test, y_test)

print(f"Total training samples: {len(train_dataset)}")
print(f"Total test samples: {len(test_dataset)}")



train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("DataLoaders created.")

Loading labels...
Loading and concatenating training spectra (this may take 1-2 minutes)...


Training data:   0%|          | 0/100 [00:00<?, ?it/s]

Loading and concatenating test spectra...


Test data    :   0%|          | 0/2 [00:00<?, ?it/s]

Raw X_train shape: (50000, 40002), Raw y_train shape: (50000,)
Raw X_test shape: (20000, 40002), Raw y_test shape: (20000,)
Scaling data (this may take a moment)...
Data loading and preprocessing complete.

Creating PyTorch Datasets...
Total training samples: 50000
Total test samples: 20000
DataLoaders created.


In [10]:
!ls -l *.csv


-rw-rw-r-- 1 user6 user6 63709 Nov 16 00:17 test_labels.csv


In [11]:
!mv 'test_labels (1).csv' test_labels.csv

mv: cannot stat 'test_labels (1).csv': No such file or directory


In [12]:
!ls -l test_labels.csv

-rw-rw-r-- 1 user6 user6 63709 Nov 16 00:17 test_labels.csv


In [14]:
class CNN1D(nn.Module):

    def __init__(self, num_classes=12):
        super(CNN1D, self).__init__()
        
        # CNN Feature Extractor
        # Input (Batch, 1, 40002)
        
    
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4), # 40002 -> 10000
            nn.Dropout(0.3) 
        )
        
        # Block 2
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4), # 10000 -> 2500
            nn.Dropout(0.3) # Increased dropout
        )
        
        # Block 3 
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=10, stride=10), # 2500 -> 250
            nn.Dropout(0.4) # More dropout
        )
        
        #  Classifier # Flattened size = 128 filters * 250 features = 32000
    
        self.flattened_size = 128 * 250 
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flattened_size, 256),
            nn.ReLU(),
            nn.Dropout(0.6), # Increased dropout
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.6), # Increased dropout
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Pass through all 3 convolutional blocks
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        # Classify
        x = self.classifier(x)
        return x
        
model = CNN1D(num_classes=NUM_CLASSES).to(device)
print("NEW V2 Model architecture created and moved to GPU.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(model)

NEW V2 Model architecture created and moved to GPU.
Total parameters: 8,261,964
CNN1D(
  (conv_block1): Sequential(
    (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.3, inplace=False)
  )
  (conv_block2): Sequential(
    (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.3, inplace=False)
  )
  (conv_block3): Sequential(
    (0): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=10, stride=10, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.4, inplace=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32000, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.6, inplace=False)

In [15]:
def train_model(model, train_loader, criterion, optimizer, epoch):
    model.train()  # Set model to training mode
    total_loss = 0
    start_time = time.time()
    
    # Use tqdm for a progress bar on the training loop
    for i, (spectra, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        spectra = spectra.to(device)
        labels = labels.to(device)
        
        # Training Steps 
        outputs = model(spectra)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        
        total_loss += loss.item()
            
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'--- Epoch {epoch+1} Summary ---')
    print(f'Average Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s')

def evaluate_model(model, test_loader, criterion):
    print("\nEvaluating model on test data...")
    model.eval()  # Set model to evaluation mode
    
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad(): 
        for spectra, labels in tqdm(test_loader, desc="Evaluating"):
            spectra = spectra.to(device)
            labels = labels.to(device)
            
            outputs = model(spectra)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f'Test Loss: {avg_loss:.4f}')
    print(f'Test Accuracy: {accuracy * 100:.2f}%')
    
    print("\n--- Classification Report ---")
    # Note: The test set only has 11 classes (0-10). Class 11 is missing.
    # The 'labels' list ensures the report is formatted correctly.
    target_names = [f"Class {i+1}" for i in range(NUM_CLASSES)]
    print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=target_names, digits=4, zero_division=0))

In [16]:
# Loss and Optimizer

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

print("\nStarting training (with V2 Model, Weighted Loss, and Scheduler)...")
total_start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # re-use the same train_model function from Cell 6
    train_model(model, train_loader, criterion, optimizer, epoch)
    
    
    scheduler.step()
    

    current_lr = optimizer.param_groups[0]['lr']
    print(f"--- End of Epoch {epoch+1}, Current LR: {current_lr} ---")
    
total_end_time = time.time()
print(f"\n--- Training Finished ---")
print(f"Total training time: {(total_end_time - total_start_time)/60:.2f} minutes")

evaluate_model(model, test_loader, criterion)


Starting training (with V2 Model, Weighted Loss, and Scheduler)...


Epoch 1/30:   0%|          | 0/196 [00:01<?, ?it/s]

--- Epoch 1 Summary ---
Average Training Loss: 1.9074, Time: 30.66s
--- End of Epoch 1, Current LR: 0.0005 ---


Epoch 2/30:   0%|          | 0/196 [00:01<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
  File "/opt/anaconda3/envs/odenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/datal

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
  File "/opt/anaconda3/envs/odenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/datal

--- Epoch 30 Summary ---
Average Training Loss: 0.2733, Time: 31.64s
--- End of Epoch 30, Current LR: 5e-06 ---

--- Training Finished ---
Total training time: 15.36 minutes

Evaluating model on test data...


Evaluating:   0%|          | 0/79 [00:01<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
  File "/opt/anaconda3/envs/odenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f251d46e560>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/odenv/lib/python3.10/site-packages/torch/utils/data/datal

Test Loss: 2.5597
Test Accuracy: 69.47%

--- Classification Report ---
              precision    recall  f1-score   support

     Class 1     0.7127    0.4250    0.5325      3146
     Class 2     0.9859    0.8939    0.9376      3129
     Class 3     0.3736    0.1252    0.1876       543
     Class 4     0.8125    0.6837    0.7425      1603
     Class 5     0.2338    0.4532    0.3084      1048
     Class 6     0.5487    1.0000    0.7086      1589
     Class 7     0.0693    0.0732    0.0712       560
     Class 8     0.9755    0.9528    0.9640      1546
     Class 9     0.7865    0.9613    0.8652      3127
    Class 10     0.4308    0.9864    0.5996       514
    Class 11     0.8709    0.4707    0.6111      3195
    Class 12     0.0000    0.0000    0.0000         0

    accuracy                         0.6946     20000
   macro avg     0.5667    0.5855    0.5440     20000
weighted avg     0.7480    0.6946    0.6923     20000

